In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from tqdm import tqdm
import math
import os
import re
import tabulate
from IPython.display import display, Markdown
import scipy.optimize as opt

# Custom packages 
import instrumental_programs as iv
from iv_lp import *

# Personalized packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter
from coverage_functions import coverage_calculator, plot_time_series, plot_time_spacing


Preemptively set new Pandas option, also set matplotlib to close

In [ ]:
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

Allow reloading of custom Python classes without resetting kernel

In [ ]:
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged
%store -r before_after_details_true
%store -r restaurants_by_4m_coverage
%store -r time_differences
%store -r time_differences_details

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

In [ ]:
%store -r df1

In [ ]:
all_bins = {}

bins1 = [0, .2, .4, .5, .7, .9]
bins2 = [0, .2, .4, .5, .7, .9]
bins3 = [0, .2, .4, .5, .7, .9]
bins4 = [0, .2, .4, .5, .7, .9]
bins = [bins1, bins2, bins3, bins4]
all_bins['LQ5EH4BKGV61T'] = bins

bins1 = [0, .2, .4, .5, .7, .9]
bins2 = [0, .2, .4, .5, .7, .9]
bins3 = [0, .2, .4, .5, .7, .9]
bins4 = [0, .2, .4, .5, .7, .9]
bins = [bins1, bins2, bins3, bins4]
all_bins['LQ5EH4BKGV61T'] = bins


def bin(bins, loc_id='LQ5EH4BKGV61T', freq='W', thin=None):
       
    df = sales_and_menu_data[loc_id]
    promo_datetime = before_after_details_true.loc[loc_id, 'cross_over_date']
    two_months_before = promo_datetime - pd.DateOffset(weeks=8)
    two_months_after = promo_datetime + pd.DateOffset(weeks=8)

    weekly_intervals = pd.date_range(two_months_before, two_months_after, freq='W-FRI', inclusive='both')
    if thin:

    weekly_abf_pct_list = []
    weekly_total_sales_list = []
    weekly_abf_cat_list = []

    for i in range(len(weekly_intervals)-1):
        start = weekly_intervals[i]
        end = weekly_intervals[i + 1]

        weekly_df = df.loc[start:end]

        total_sales = (weekly_df
                        ['item_quantity']
                        .sum())

        abf_sales = (weekly_df
                        .query('is_plant_based == "No"')
                        ['item_quantity']
                        .sum()
                        )

        abf_proportion = abf_sales / total_sales if total_sales > 0 else 999 # 999 is a dummy number

        weekly_total_sales_list.append(total_sales)
        weekly_abf_pct_list.append(abf_proportion)

        weekly_abf_cat = np.array(weekly_abf_cat_list)
        weekly_abf_pct = np.array(weekly_abf_pct_list)

        before_total = (df
                        .loc[two_months_before:promo_datetime]
                        ['item_quantity']
                        .sum())
        after_total = (df
                        .loc[promo_datetime:two_months_after]
                        ['item_quantity']
                        .sum())
        before_abf_pct = (df
                .loc[two_months_before:promo_datetime]
                .pipe(lambda df: print(df.shape) or df)
                .query('is_plant_based == "No"')
                ['item_quantity']
                .sum()
                /before_total)
        after_abf_pct = (df
                .loc[promo_datetime:two_months_after]
                .query('is_plant_based == "No"')
                ['item_quantity']
                .sum()
                /after_total)

        print(before_total, after_total)
        round(before_abf_pct, 3), round(after_abf_pct, 3)


        
        return len(bins), ps, np.digitize(ps, bins)



In [ ]:
def proportion(loc_id, freq='W-FRI', thinfreq=None, thintype=0):
    # Day of the week: Monday = 0, ..., Sunday = 6
    
    df = sales_and_menu_data[loc_id]
    promo_datetime = before_after_details_true.loc[loc_id, 'cross_over_date']
    two_months_before = promo_datetime - pd.DateOffset(weeks=8)
    two_months_after = promo_datetime + pd.DateOffset(weeks=8)
    df = df[two_months_before:two_months_after]
    
    if thinfreq:
        if thinfreq == 'day':
            df[df.index.dayofweek == thintype]
        elif thinfreq == 'hour':
            df[df.index.hour == thintype]
        else:
            raise Exception('Invalid thintype. Please choose a day of the week or hour of the day.')
    
    abf_resampled = (df
                     .query('is_plant_based == "No"')
                     .resample(freq)
                     ['item_quantity']
                     .sum())
    total_resampled = (df
                     .resample(freq)
                     ['item_quantity']
                     .sum())
    abf_proportion = abf_resampled / total_resampled
    
    return {
        'total_count': total_resampled,
        'abf_proportion': abf_proportion,
        'promo_datetime': promo_datetime
    }


def normal_bins_4(props):
    mean = props.mean()
    sd = props.std()
    if mean - sd < 0 or 1 < mean + sd:
        raise Exception("Variance too high for normal binning.")
    return [0, mean - sd, mean, mean + sd, 1]


def normal_bins_6(props):
    mean = props.mean()
    sd = props.std()
    if mean - 2*sd < 0 or 1 < mean + 2*sd:
        raise Exception("Variance too high for normal binning.")
    return [0, mean - 2*sd, mean - sd, mean, mean + sd, mean + 2*sd, 1]


def max_min_bins_4(props, epsilon=0.01):
    mean = props.mean()
    maxi = props.max() + epsilon
    mini = props.min() - epsilon
    return [mini, (mini+mean)/2, mean, (maxi+mean)/2, maxi]


def bin(abf_proportion, bins):

    raw_categories = np.digitize(abf_proportion, bins)

    abf_categories = pd.Series(data=raw_categories, 
                               index=pd.DatetimeIndex(abf_proportion.index))

    return {
        'abf_category': abf_categories,
        'nbin': len(bins)-1,
        'bins': bins,
    }


# Example
bins1 = [0, 0.2, 0.4, 0.5, 0.7, 0.9]
bins2 = [0, 0.2, 0.4, 0.5, 0.7, 0.9]
bins3 = [0, 0.2, 0.4, 0.5, 0.7, 0.9]
bins4 = [0, 0.2, 0.4, 0.5, 0.7, 0.9]
all_bins = {: [bins1, bins2, bins3, bins4]}

low_data_loc_ids = []

for loc_id in low_data_loc_ids:
    summary = proportion(loc_id)
    summary['total_count']
    prop = summary['abf_proportion']
    summary['promo_datetime']

result = bin(prop, bin)


In [ ]:
result['abf_category'].loc[:result['promo']]

In [ ]:
def restaurant_sales_lp():
    # Define variables
    graph = [('T', 'Z'), ('Z', 'Y')]
    instruments = set()
    measurements = {'Y'}
    unobserved = {'Z'}
    
    
    # Discretize the sales proportion into 100 categories (0.00 to 0.99 in increments of 0.01)
    bins = []
    nbins = len(bins)
    cardinalities = {'T': 2, 'Y': nbins}
    
    # Create a placeholder distribution; you should replace this with your actual estimates
    dist_dims = ['T', 'Z', 'Y']
    
    # Measurement error assumptions
    assumptions = [
        {
            'type': 'error_bound',
            'parent': 'Z',
            'child': 'Y',
            'kwargs': {'epsilon': 0.01},  # Adjust based on the expected measurement error
        },
        {
            'type': 'monotonicity',
            'parent': 'Z',
            'child': 'Y',
            'kwargs': {},
        },
    ]
    
    # Build the linear program
    linprog_args = iv.build_lp(graph,
                               cardinalities,
                               instruments,
                               measurements,
                               unobserved,
                               dist,
                               dist_dims,
                               target_var='Y',
                               intervention={'A': 0},
                               intervention2={'A': 1},
                               assumptions=assumptions,
                               n=1000,
                               alpha=0.05)
    #print(linprog_args)
    print(opt.linprog(*linprog_args))


restaurant_sales_lp()

In [ ]:
def test_iv():
    graph = [('Z', 'A'), ('A', 'Y')]
    instruments = {'Z'}
    measurements = {'A', 'Y'}
    unobserved = set()
    cardinalities = {'Z': 2, 'A': 2, 'Y': 2}

    dist = np.array([[[0.125, 0.125], [0.125, 0.125]], [[0.125, 0.125], [0.125, 0.125]]])
    dist_dims = ['Z', 'A', 'Y']

    assumptions = [
        {
            'type': 'positive_effect',
            'parent': 'A',
            'child': 'Y',
            'kwargs': {
                'epsilon': 0.01
            },
        },
    ]

    linprog_args = iv.build_lp(graph,
                               cardinalities,
                               instruments,
                               measurements,
                               unobserved,
                               dist,
                               dist_dims,
                               target_var='Y',
                               intervention={'A': 0},
                               intervention2={'A': 1},
                               assumptions=assumptions,
                               n=1000,
                               alpha=0.05)
    #print(linprog_args)
    print(opt.linprog(*linprog_args))

test_iv()

In [ ]:
def cost_fun(v):
    return 1*(v < 3)

promo = np.array(weekly_abf_pct.shape[0]//2 * [0] + weekly_abf_pct.shape[0]//2 * [1])
T = promo[weekly_abf_cat != -1]
Y = weekly_abf_cat[weekly_abf_cat != -1]

weights = np.ones(len(T))

pztm = np.zeros((2, 2, chunks))  # Shape: (2, 2, chunks)
for t in range(2):
    for y in range(chunks):
        # Since there's no IV, we distribute the probabilities equally across the two IV levels
        pztm[0, t, y] = np.average((T == t) & (Y == y), weights=weights) / 2
        pztm[1, t, y] = np.average((T == t) & (Y == y), weights=weights) / 2

sample_sizes = np.zeros(2)
for t in range(2):
    sample_sizes[t] = np.sum(weights[T == t])**2 / np.sum(weights[T == t]**2)

assumptions = [
    {"EB1":True,"EB2":False,"Mon":False,"Sym":False,"Exp":False},
    {"EB1":True,"EB2":False,"Mon":False,"Sym":False,"Exp":True},
    {"EB1":True,"EB2":False,"Mon":True,"Sym":False,"Exp":True},
    {"EB1":True,"EB2":False,"Mon":False,"Sym":True,"Exp":True},
]

# Define epsilon grid
S = 41
grid_epsilons = np.linspace(0,0.04,S)

# Empty arrays for point estimated bounds and confidence intervals
grid_point_bounds = np.zeros((len(assumptions),S,2))
grid_ci_bounds = np.zeros((len(assumptions),S,2))
            
for k,ass in enumerate(assumptions):
    print(ass)
    
    for s in tqdm(list(range(len(grid_epsilons)))):
        eps = grid_epsilons[s]

        sym_const = ass["Sym"]
        mon_const = ass["Mon"]
        exp_const = ass["Exp"]
        pos_effect_of_truth = False
        pos_effect_of_treatment = True

        eb_args = []
        if ass["EB1"]:
            eb_args.append((0.0,eps))
        if ass["EB2"]:
            eb_args.append((1.0,eps/10))

        lb,ub,lb_res,ub_res,lp_args = get_iv_bounds(pztm,
                                    0,
                                    eb_args,
                                    err_const=True,
                                    mon_const=mon_const,
                                    sym_const=sym_const,
                                    exp_const=exp_const,
                                    pos_effect_of_treatment=pos_effect_of_treatment,
                                    pos_effect_of_truth=False,
                                    parameter="ate",
                                    alpha=0.99999999,
                                    n=100000000,
                                    fun=cost_fun,
                                    x0=None)

        grid_point_bounds[k,s,0] = lb
        grid_point_bounds[k,s,1] = ub
        
        lb,ub,lb_res,ub_res,lp_args = get_iv_bounds(pztm,
                                    0,
                                    eb_args,
                                    err_const=True,
                                    mon_const=mon_const,
                                    sym_const=sym_const,
                                    exp_const=exp_const,
                                    pos_effect_of_treatment=pos_effect_of_treatment,
                                    pos_effect_of_truth=False,
                                    parameter="ate",
                                    alpha=0.05,
                                    n=None,
                                    fun=cost_fun,
                                    x0=None,
                                    sample_sizes=sample_sizes)

        grid_ci_bounds[k,s,0] = lb
        grid_ci_bounds[k,s,1] = ub